# FOI – 230416001 – Statistics on first disposal of a C100 Application

## 1. Import packages and set options 
<a name="import_packages"></a>

In [1]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [2]:
#this is the database we will be extracting from
database = "familyman_dev_v3"
derived = "familyman_derived_dev_v3"

#this extracts the latest snapshot from athena
#get_snapshot_date = f"SELECT mojap_snapshot_date from {database}.events order by mojap_snapshot_date desc limit 1"
#snapshot_date = str(pydb.read_sql_query(get_snapshot_date)['mojap_snapshot_date'].values[0])

#this extracts the February snapshot from athena
snapshot_date = "2023-02-09"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 4
pub_year = 2022

## Extract date of issue

In [4]:
pydb.create_temp_table(
f""" 
SELECT 
  case_number,
  CAST(value AS Date) AS case_DOI
FROM 
  {database}.case_fields AS f
 
WHERE
  field_model = 'FM2C_DI'
  AND mojap_snapshot_date = DATE'{snapshot_date}'

""",

"DOI")

In [5]:
a = pydb.read_sql_query("SELECT * from __temp__.DOI")
a

,case_number,case_doi
0,LE00P00302,2000-05-03
1,LE01P00312,2001-06-05
2,LE01P00327,2001-06-12
3,LE01P00465,2001-08-16
4,LE03P00525,2003-08-26
...,...,...
1809123,MS13P00561,2013-05-13
1809124,MS16P00131,2016-11-24
1809125,MS17F00152,2017-09-25
1809126,MT02P00202,2002-07-11


## Create start date using date of issue

In [6]:
#Adding the start date to the main child apps data created in the main child apps process (so data is at child and order level)
#Only including specific order types
pydb.create_temp_table(
f""" 
SELECT 
  t1.case_number,
  t1.receipt_date,
  t1.event,
  t1.field_model,
  t1.order_type,
  t1.order_code,
  t1.order_desc,
  t1.child_role_id,
  t1.order_case_type,
  t2.case_DOI,
  CASE WHEN  t2.case_DOI is null
        THEN t1.receipt_date
       WHEN t1.field_model = 'U22_AT' AND (t2.Case_DOI<t1.Receipt_date)
        THEN t2.Case_DOI
        ELSE t1.Receipt_date END
      AS Start_date
FROM 
  {fcsq_database}.ca_apps_child AS t1
      LEFT JOIN __temp__.DOI AS t2 
       ON t1.case_number = t2.case_number
WHERE
  t1.order_code IN (29,30,31,32)

""",

"ca_apps_issue_date")

In [7]:
a = pydb.read_sql_query("SELECT * from __temp__.ca_apps_issue_date")
a

,case_number,receipt_date,event,field_model,order_type,order_code,order_desc,child_role_id,order_case_type,case_doi,start_date
0,BW19P00210,2019-11-01,12000293573,U22_AT,CAL,30,Section 8 Residence Order,10543507,P,2019-11-01,2019-11-01
1,CA19P00156,2019-06-13,16500519092,U22_AT,CAST,29,Section 8 Contact Order,10144007,P,2019-06-13,2019-06-13
2,CF19P00011,2019-01-04,16401710895,U22_AT,CAL,30,Section 8 Residence Order,9688382,P,2019-01-04,2019-01-04
3,CF19P00585,2019-04-23,16401739924,U22_AT,CAL,30,Section 8 Residence Order,9994049,P,2019-04-23,2019-04-23
4,CF19P00596,2019-04-30,26900257125,U22_AT,CAST,29,Section 8 Contact Order,10012244,P,2019-04-30,2019-04-30
...,...,...,...,...,...,...,...,...,...,...,...
2333068,DE14P00082,2014-01-31,18501143488,U22_AT,RO,30,Section 8 Residence Order,4554222,P,2014-01-31,2014-01-31
2333069,DE14P00212,2014-07-30,18501209367,U22_AT,SIO,32,Section 8 Specific Issue Order,5075485,P,2014-07-30,2014-07-30
2333070,DE14P00226,2014-08-04,18501210605,U22_AT,SIO,32,Section 8 Specific Issue Order,5087196,P,2014-08-04,2014-08-04
2333071,DE14P00287,2014-04-01,18501160466,U22_AT,CNO,29,Section 8 Contact Order,4749633,P,2014-04-02,2014-04-01


## Matching applications to disposals

In [8]:
#Matching apps data to the main child disposal data set created in the child disposals processing - matches at child and order level
#Filetering out certain order types alongside interim orders
#Ranks the earlies diposal as the first disposal
pydb.create_temp_table(
f""" 
SELECT 
  a.*,
  YEAR(d.disp_date) AS Year,
  QUARTER(d.disp_date) AS Quarter,
  MONTH(d.disp_date) AS Month,
  d.disp_date,
  date_diff('day',a.start_date, d.disp_date) AS days,
  date_diff('week',a.start_date, d.disp_date) AS weeks,
  ROW_NUMBER() OVER(PARTITION BY a.case_number, a.child_role_id, a.order_code, a.start_date
                       ORDER BY d.disp_date) 
      AS disp_rank,
  d.disposal_court,
  d.disposal_dfj,
  d.disposal_region 
FROM
  __temp__.ca_apps_issue_date a
  LEFT JOIN {fcsq_database}.ca_disps_all_children d
    ON a.case_number = d.case_number
    AND a.order_case_type = d.order_case_type
    AND a.child_role_id = d.child_role_id
WHERE 
  date_diff('day',a.start_date, d.disp_date) >= 0
  AND d.event_model NOT IN ('CPA','C21','C27','C30','C31','C33','C35B','C44A','C44B','C46A','C46B','C47A','C47C','C48A', 
                          'C48B','C48C','C49','D51','D84C','MAGEPO','MAGS37')
  AND d.disp_type_code <> 2

""",

"app_disp_match")

In [9]:
a = pydb.read_sql_query("SELECT * from __temp__.app_disp_match")
a

,case_number,receipt_date,event,field_model,order_type,order_code,order_desc,child_role_id,order_case_type,case_doi,start_date,year,quarter,month,disp_date,days,weeks,disp_rank,disposal_court,disposal_dfj,disposal_region
0,BR16P00200,2016-06-09,15201323525,U22_AT,CAL,30,Section 8 Residence Order,7013108,P,2016-06-09,2016-06-09,2016,2,6,2016-06-09,0,0,1,BROMLEY,East London Family Court,LONDON
1,BR16P00200,2016-06-09,15201323525,U22_AT,PSO,31,Section 8 Prohibited Steps Order,7013108,P,2016-06-09,2016-06-09,2016,2,6,2016-06-09,0,0,1,BROMLEY,East London Family Court,LONDON
2,BR16P00228,2016-07-13,15201331687,U22_AT,CAL,30,Section 8 Residence Order,7121161,P,2016-07-14,2016-07-13,2016,4,11,2016-11-02,112,16,1,CROYDON,East London Family Court,LONDON
3,BR16P00440,2016-12-16,15201367024,U22_AT,PSO,31,Section 8 Prohibited Steps Order,7584248,P,2016-12-16,2016-12-16,2016,4,12,2016-12-16,0,0,1,BROMLEY,East London Family Court,LONDON
4,BR16P00440,2016-12-16,15201367024,U22_AT,PSO,31,Section 8 Prohibited Steps Order,7584248,P,2016-12-16,2016-12-16,2016,4,12,2016-12-16,0,0,2,BROMLEY,East London Family Court,LONDON
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5224014,ZW22P00270,2022-02-28,33200896388,U22_AT,SIO,32,Section 8 Specific Issue Order,12799932,P,2022-02-28,2022-02-28,2022,4,12,2022-12-22,297,42,2,WEST LONDON,West London Family Court,LONDON
5224015,ZW22P00275,2022-02-28,33200896564,U22_AT,PSO,31,Section 8 Prohibited Steps Order,12800847,P,2022-02-28,2022-02-28,2022,1,3,2022-03-04,4,0,1,WEST LONDON,West London Family Court,LONDON
5224016,ZW22P00275,2022-02-28,33200896564,U22_AT,PSO,31,Section 8 Prohibited Steps Order,12800847,P,2022-02-28,2022-02-28,2022,2,6,2022-06-13,105,15,2,WEST LONDON,West London Family Court,LONDON
5224017,ZW22P00287,2022-02-28,33200897359,U22_AT,SIO,32,Section 8 Specific Issue Order,12806031,P,2022-03-01,2022-02-28,2022,4,11,2022-11-18,263,37,1,WEST LONDON,West London Family Court,LONDON


## Selecting the earliest disposal

In [10]:
#Selects the first diposal for each order type/child matched
#Multiple children can get matched to the same disposal (if the disposal has multiple child id's)
#More than one application/order type can be matched to one disposal - here we are replicating the SAS process 
pydb.create_temp_table(
f""" 
SELECT 
  *
FROM 
  __temp__.app_disp_match
WHERE
  disp_rank = 1

""",

"first_disp")

In [11]:
a = pydb.read_sql_query("SELECT * from __temp__.first_disp")
a

,case_number,receipt_date,event,field_model,order_type,order_code,order_desc,child_role_id,order_case_type,case_doi,start_date,year,quarter,month,disp_date,days,weeks,disp_rank,disposal_court,disposal_dfj,disposal_region
0,CM11P01964,2011-11-11,16701694915,U22_AT,PSO,31,Section 8 Prohibited Steps Order,1942204,P,2011-11-11,2011-11-11,2011,4,12,2011-12-21,40,5,1,Chelmsford,Chelmsford DFJ,SOUTH EAST
1,CM11P01969,2011-11-14,16701695317,U22_AT,CNO,29,Section 8 Contact Order,1946386,P,2011-11-14,2011-11-14,2012,1,1,2012-01-11,58,8,1,Colchester,Chelmsford DFJ,SOUTH EAST
2,CM11P02155,2011-12-09,16701704759,U22_AT,RO,30,Section 8 Residence Order,2036878,P,2011-12-09,2011-12-09,2011,4,12,2011-12-12,3,0,1,Chelmsford,Chelmsford DFJ,SOUTH EAST
3,CM11P02226,2011-12-20,16701708861,U22_AT,CNO,29,Section 8 Contact Order,2072204,P,2011-12-21,2011-12-20,2012,1,2,2012-02-27,69,9,1,Chelmsford,Chelmsford DFJ,SOUTH EAST
4,CM11P10107,2011-03-14,45000067474,U22_AT,RO,30,Section 8 Residence Order,1136319,P,2011-03-14,2011-03-14,2011,2,4,2011-04-14,31,4,1,Essex,Chelmsford DFJ,SOUTH EAST
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1807161,LS10P08248,2010-04-20,24302290125,U22_AT,CNO,29,Section 8 Contact Order,24300375617,P,2010-04-20,2010-04-20,2011,3,7,2011-07-27,463,66,1,Leeds,Leeds DFJ,NORTH EAST
1807162,LS10P08317,2010-05-20,24302375657,U22_AT,PSO,31,Section 8 Prohibited Steps Order,341381,P,2010-05-20,2010-05-20,2010,2,6,2010-06-03,14,2,1,Leeds,Leeds DFJ,NORTH EAST
1807163,LS10P08396,2010-06-21,24302398258,U22_AT,CNO,29,Section 8 Contact Order,396523,P,2010-06-21,2010-06-21,2010,2,6,2010-06-21,0,0,1,Leeds,Leeds DFJ,NORTH EAST
1807164,LS10P08497,2010-08-09,24302431171,U22_AT,PSO,31,Section 8 Prohibited Steps Order,501794,P,2010-08-09,2010-08-09,2010,4,12,2010-12-20,133,19,1,Leeds,Leeds DFJ,NORTH EAST


## Creation of main timeliess dataset, deletion of dup apps

In [12]:
#Here we get rid of any instances of an application of the same order type for the same child, and only the earliest one is selected
pydb.create_temp_table(
f""" 
WITH dup_app_type AS
 (SELECT 
   *,
   ROW_NUMBER() OVER(PARTITION BY case_number, child_role_id, order_code
                       ORDER BY start_date, case_number, child_role_id, order_code) 
     AS dup_app_rank
 FROM 
   __temp__.first_disp
 )

SELECT
  *,
  cast((EVENT / 100000000) as int)  AS EVENT_COURT
FROM
  dup_app_type
WHERE
  dup_app_rank = 1

""",

"ca_timeliness_all")

In [13]:
count = pydb.read_sql_query("SELECT count(*) as count from __temp__.ca_timeliness_all")
count

,count
0,1786148


In [5]:
a = pydb.read_sql_query("SELECT * from __temp__.ca_timeliness_all limit 10")
a

,case_number,receipt_date,event,field_model,order_type,order_code,order_desc,child_role_id,order_case_type,case_doi,start_date,year,quarter,month,disp_date,days,weeks,disp_rank,disposal_court,disposal_dfj,disposal_region,dup_app_rank,event_court
0,DE16P00572,2016-10-28,18501453520,U22_AT,CAL,30,Section 8 Residence Order,7429294,P,2016-10-28,2016-10-28,2016,4,11,2016-11-02,5,0,1,DERBY,Derby DFJ,MIDLANDS,1,185
1,DE18P00094,2018-02-16,18501629748,U22_AT,CAST,29,Section 8 Contact Order,8802349,P,2018-02-19,2018-02-16,2018,1,3,2018-03-28,40,5,1,DERBY,Derby DFJ,MIDLANDS,1,185
2,DE18P00678,2018-11-19,18501729258,U22_AT,CAL,30,Section 8 Residence Order,9577607,P,2018-11-20,2018-11-19,2019,2,5,2019-05-30,192,27,1,DERBY,Derby DFJ,MIDLANDS,1,185
3,MA08P00248,2008-07-22,26201768032,U22_AT,RO,30,Section 8 Residence Order,26200296462,P,2008-07-22,2008-07-22,2008,3,7,2008-07-22,0,0,1,Manchester,Manchester DFJ,NORTH WEST,1,262
4,MA12P00205,2012-03-20,26202757457,U22_AT,RO,30,Section 8 Residence Order,2364040,P,2012-03-20,2012-03-20,2012,2,5,2012-05-08,49,7,1,Manchester,Manchester DFJ,NORTH WEST,1,262
5,MA15P00233,2015-02-12,26203654361,U22_AT,CAST,29,Section 8 Contact Order,5615452,P,2015-02-12,2015-02-12,2015,1,3,2015-03-17,33,4,1,WIGAN,Manchester DFJ,NORTH WEST,1,262
6,IG07P01024,2007-11-28,23100193243,U22_AT,PSO,31,Section 8 Prohibited Steps Order,23100060370,P,2007-11-28,2007-11-28,2007,4,11,2007-11-28,0,0,1,Ilford,Ilford,LONDON,1,231
7,IP07P00301,2006-11-20,23300343364,U22_AT,CNO,29,Section 8 Contact Order,23300085658,P,2006-11-20,2006-11-20,2007,2,6,2007-06-27,219,31,1,Ipswich,Norwich DFJ,SOUTH EAST,1,233
8,NE03P01389,2003-09-01,27800474083,U22_AT,SIO,32,Section 8 Specific Issue Order,27800083415,P,2003-09-01,2003-09-01,2003,3,9,2003-09-01,0,0,1,Newcastle Upon Tyne,Newcastle upon Tyne DFJ,NORTH EAST,1,278
9,NE05P01553,2005-10-05,27800900294,U22_AT,CNO,29,Section 8 Contact Order,27800113736,P,2005-10-05,2005-10-05,2006,1,1,2006-01-13,100,14,1,Newcastle Upon Tyne,Newcastle upon Tyne DFJ,NORTH EAST,1,278


## Adding harm flags to c100 applications

In [7]:
pydb.create_temp_table(
f""" 

SELECT DISTINCT t1.*,
t2.harm_alleged
from __temp__.ca_timeliness_all t1
LEFT JOIN {derived}.cases_derived t2
ON t1.CASE_NUMBER = t2.CASE_NUMBER 


""",

"c100_apps_harm_flag")

In [8]:
count = pydb.read_sql_query("SELECT count(*) as count from __temp__.c100_apps_harm_flag ")
count

,count
0,1786149


In [9]:
a = pydb.read_sql_query("SELECT * from __temp__.c100_apps_harm_flag limit 10")
a

,case_number,receipt_date,event,field_model,order_type,order_code,order_desc,child_role_id,order_case_type,case_doi,start_date,year,quarter,month,disp_date,days,weeks,disp_rank,disposal_court,disposal_dfj,disposal_region,dup_app_rank,event_court,harm_alleged
0,SS05P00361,2005-03-09,32900345354,U22_AT,PSO,31,Section 8 Prohibited Steps Order,32900084408,P,2005-03-09,2005-03-09,2005,1,3,2005-03-09,0,0,1,Southend,Chelmsford DFJ,SOUTH EAST,1,329,Y
1,KH12P05583,2012-10-16,23900959356,U22_AT,SIO,32,Section 8 Specific Issue Order,3056068,P,2012-10-16,2012-10-16,2012,4,10,2012-10-17,1,0,1,Kingston-Upon-Hull,Kingston-upon-Hull DFJ,NORTH EAST,1,239,Y
2,RR13P00030,2013-07-31,52200021088,U22_AT,RO,30,Section 8 Residence Order,4016110,P,2013-07-31,2013-07-31,2013,3,9,2013-09-04,35,5,1,Rotherham,Sheffield DFJ,NORTH EAST,1,522,Y
3,BO09P01356,2009-11-02,14000577898,U22_AT,CNO,29,Section 8 Contact Order,14000114992,P,2009-11-02,2009-11-02,2010,1,3,2010-03-24,142,20,1,Bow,East London Family Court,LONDON,1,140,N
4,NE03P01814,2003-11-18,27800674513,U22_AT,CNO,29,Section 8 Contact Order,27800086798,P,2003-11-18,2003-11-18,2003,4,12,2003-12-15,27,3,1,Newcastle Upon Tyne,Newcastle upon Tyne DFJ,NORTH EAST,1,278,N
5,WM13P00727,2013-10-11,37000428432,U22_AT,PSO,31,Section 8 Prohibited Steps Order,4243562,P,2013-10-11,2013-10-11,2013,4,10,2013-10-14,3,0,1,Weston-Super-Mare,Bristol DFJ,SOUTH WEST,1,370,N
6,NE18P02529,2018-12-18,27803650301,U22_AT,CAST,29,Section 8 Contact Order,9656879,P,2018-12-18,2018-12-18,2019,2,6,2019-06-04,168,24,1,NEWCASTLE UPON TYNE,Newcastle upon Tyne DFJ,NORTH EAST,1,278,N
7,BM15P09234,2015-11-19,12705428837,U22_AT,CAST,29,Section 8 Contact Order,6425045,P,2015-11-23,2015-11-19,2016,1,1,2016-01-20,62,8,1,BIRMINGHAM,Birmingham DJF,MIDLANDS,1,127,N
8,ED04P02075,2004-12-14,19400213607,U22_AT,RO,30,Section 8 Residence Order,19400068992,P,2004-12-14,2004-12-14,2005,1,1,2005-01-24,41,5,1,Edmonton,East London Family Court,LONDON,1,194,N
9,LU11P03553,2011-08-18,25801015136,U22_AT,PSO,31,Section 8 Prohibited Steps Order,1655792,P,2011-08-18,2011-08-18,2011,3,8,2011-08-26,8,1,1,Luton,Luton DFJ,SOUTH EAST,1,258,N


## Table 1i

In [25]:
pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year
""",

"table_1i")

In [26]:
table_1i = pydb.read_sql_query("SELECT * from __temp__.table_1i order by year")
table_1i

,case_type,year,count,mean_duration
0,Children Act (Private),2015,70073,13.8
1,Children Act (Private),2016,78335,13.2
2,Children Act (Private),2017,83880,13.7
3,Children Act (Private),2018,86773,14.8
4,Children Act (Private),2019,96914,16.0
5,Children Act (Private),2020,86793,18.9
6,Children Act (Private),2021,98125,26.2
7,Children Act (Private),2022,97220,30.1


## Table 1ii

In [27]:
pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year,
  disposal_region
""",

"table_1ii")

In [28]:
table_1ii = pydb.read_sql_query("SELECT * from __temp__.table_1ii order by year, region")
table_1ii

,case_type,year,region,count,mean_duration
0,Children Act (Private),2015,LONDON,8557,15.9
1,Children Act (Private),2015,MIDLANDS,13189,16.8
2,Children Act (Private),2015,NORTH EAST,10762,11.9
3,Children Act (Private),2015,NORTH WEST,9060,16.4
4,Children Act (Private),2015,SOUTH EAST,15416,10.6
5,Children Act (Private),2015,SOUTH WEST,8661,11.3
6,Children Act (Private),2015,WALES,4428,16.3
7,Children Act (Private),2016,LONDON,8991,13.4
8,Children Act (Private),2016,MIDLANDS,16516,15.0
9,Children Act (Private),2016,NORTH EAST,11625,11.6


## Table 1iii

In [29]:
pydb.create_temp_table(
f""" 
 SELECT 
  'Children Act (Private)' as Case_type,
  Year,
  'Essex & Suffolk DFJ' AS dfj, 
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.ca_timeliness_all
WHERE
  order_case_type = 'P' 
  AND YEAR > 2014 
  AND YEAR < 2023
  AND disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')
GROUP BY
  year,
  disposal_region
ORDER BY
  year
""",

"table_1iii")

In [30]:
table_1iii = pydb.read_sql_query("SELECT * from __temp__.table_1iii")
table_1iii

,case_type,year,dfj,region,count,mean_duration
0,Children Act (Private),2015,Essex & Suffolk DFJ,SOUTH EAST,2628,11.5
1,Children Act (Private),2016,Essex & Suffolk DFJ,SOUTH EAST,3178,13.4
2,Children Act (Private),2017,Essex & Suffolk DFJ,SOUTH EAST,3027,13.9
3,Children Act (Private),2018,Essex & Suffolk DFJ,SOUTH EAST,3476,13.7
4,Children Act (Private),2019,Essex & Suffolk DFJ,SOUTH EAST,4114,13.9
5,Children Act (Private),2020,Essex & Suffolk DFJ,SOUTH EAST,4884,15.4
6,Children Act (Private),2021,Essex & Suffolk DFJ,SOUTH EAST,3941,24.7
7,Children Act (Private),2022,Essex & Suffolk DFJ,SOUTH EAST,4351,30.4


## Table 2i

In [31]:
pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P' 
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year
""",

"table_2i")

In [32]:
table_2i = pydb.read_sql_query("SELECT * from __temp__.table_2i order by year")
table_2i

,case_type,year,count,mean_duration
0,Children Act (Private),2015,33511,11.1
1,Children Act (Private),2016,38060,11.1
2,Children Act (Private),2017,41788,11.6
3,Children Act (Private),2018,44847,12.6
4,Children Act (Private),2019,51561,13.6
5,Children Act (Private),2020,51724,16.0
6,Children Act (Private),2021,61274,23.5
7,Children Act (Private),2022,62667,27.4


## Table 2ii

In [33]:
pydb.create_temp_table(
f""" 
 SELECT
  'Children Act (Private)' as Case_type,
  Year,
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P'
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
GROUP BY
  year,
  disposal_region
""",

"table_2ii")

In [34]:
table_2ii = pydb.read_sql_query("SELECT * from __temp__.table_2ii order by year, region")
table_2ii

,case_type,year,region,count,mean_duration
0,Children Act (Private),2015,LONDON,3980,11.8
1,Children Act (Private),2015,MIDLANDS,6016,14.5
2,Children Act (Private),2015,NORTH EAST,5531,8.5
3,Children Act (Private),2015,NORTH WEST,4312,13.4
4,Children Act (Private),2015,SOUTH EAST,7402,8.3
5,Children Act (Private),2015,SOUTH WEST,4420,9.5
6,Children Act (Private),2015,WALES,1850,16.0
7,Children Act (Private),2016,LONDON,4519,9.9
8,Children Act (Private),2016,MIDLANDS,8023,13.5
9,Children Act (Private),2016,NORTH EAST,6014,8.9


## Table 2iii

In [35]:
pydb.create_temp_table(
f""" 
 SELECT 
  'Children Act (Private)' as Case_type,
  Year,
  'Essex & Suffolk DFJ' AS dfj, 
  disposal_region AS region,
  COUNT(*) AS count,
  ROUND(AVG ((days)/7),1) as mean_duration
FROM
  __temp__.c100_apps_harm_flag
WHERE
  order_case_type = 'P' 
  AND harm_alleged = 'Y'
  AND YEAR > 2014 
  AND YEAR < 2023
  AND disposal_court in ('SOUTHEND', 'BURY ST. EDMUNDS', 'COLCHESTER', 'CHELMSFORD', 'IPSWICH')
GROUP BY
  year,
  disposal_region
ORDER BY
  year
""",

"table_2iii")

In [36]:
table_2iii = pydb.read_sql_query("SELECT * from __temp__.table_2iii")
table_2iii

,case_type,year,dfj,region,count,mean_duration
0,Children Act (Private),2015,Essex & Suffolk DFJ,SOUTH EAST,1243,10.4
1,Children Act (Private),2016,Essex & Suffolk DFJ,SOUTH EAST,1505,13.7
2,Children Act (Private),2017,Essex & Suffolk DFJ,SOUTH EAST,1535,12.7
3,Children Act (Private),2018,Essex & Suffolk DFJ,SOUTH EAST,1725,11.9
4,Children Act (Private),2019,Essex & Suffolk DFJ,SOUTH EAST,2365,12.3
5,Children Act (Private),2020,Essex & Suffolk DFJ,SOUTH EAST,3237,13.1
6,Children Act (Private),2021,Essex & Suffolk DFJ,SOUTH EAST,2702,21.5
7,Children Act (Private),2022,Essex & Suffolk DFJ,SOUTH EAST,2931,28.0


## Create Final Output

In [37]:
# requires pandas package
import pandas as pd
import xlsxwriter
import boto3, io, os, pathlib

In [38]:
#creating a panda dataframe for all 4 tables (1A, 1B, 2A, 2B)
df1 = pd.DataFrame(table_1i)

df2 = pd.DataFrame(table_1ii)

df3 = pd.DataFrame(table_1iii)

df4 = pd.DataFrame(table_2i)

df5 = pd.DataFrame(table_2ii)

df6 = pd.DataFrame(table_2iii)

In [40]:
with io.BytesIO() as xlsx_data:
    with pd.ExcelWriter(xlsx_data) as writer:
        
        df1.to_excel(writer, sheet_name='T1a', index=False)

        df2.to_excel(writer, sheet_name='T1b', index=False)

        df3.to_excel(writer, sheet_name='T1c', index=False)

        df4.to_excel(writer, sheet_name='T2a', index=False)
        
        df5.to_excel(writer, sheet_name='T2b', index=False)
        
        df6.to_excel(writer, sheet_name='T2c', index=False)
    ## Upload the in-memory data to S3
    s3 = boto3.resource("s3")
    bucket = s3.Bucket("alpha-family-data")
    object_name = "FOIs/FOI 230416001 – working table.xlsx"
    s3.Bucket("alpha-family-data").put_object(Key=object_name, Body=xlsx_data.getvalue())